# Merged analysis of HAL software mentions (Softcite + Baguette + Bio2Flow)

This notebook analyses the **merged** software-mention predictions for PEPR Santé Numérique
publications. Each `*.software.json` file in `data/merge/` was produced by merging the outputs of
three tools into a single file per paper, taking the **native Softcite** output as the reference and
**adding** the mentions that Baguette and Bio2Flow detected but Softcite missed.

Because the files carry the same Softcite schema, this notebook reuses the same analysis pipeline as
`softcite_analysis.ipynb`. On top of that, it exploits the two extra fields added during the merge:

- `origins` on each mention — the list of tools (`softcite`, `baguette`, `bio2flow`) that produced it;
- `merged_from` at the document level — how many mentions of each tool ended up in the file.

The main additions over the original notebook are a **per-source / complementarity** analysis
(section 3) and a **dedicated dataset analysis** (section 7), because the merge also keeps the
`dataset` mentions produced by Baguette.

## 0 — Setup

In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Merged predictions live in <repo>/data/merge; this notebook lives in notebooks/.
DATA_DIR = Path("data/merge")
if not DATA_DIR.exists():
    DATA_DIR = Path("../data/merge")
assert DATA_DIR.exists(), f"Could not find {DATA_DIR.resolve()}"

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# --- chart style -------------------------------------------------------------
THEME = "light"
STYLE = {
    "light": dict(surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9", axis="#c3c2b7",
                  series=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]),
}[THEME]
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
_axis = dict(showgrid=True, gridcolor=STYLE["grid"], gridwidth=1, showline=True, linecolor=STYLE["axis"], linewidth=1,
             zeroline=False, ticks="", tickfont=dict(color=STYLE["muted"]),
             title=dict(font=dict(color=STYLE["ink2"])), automargin=True)
pio.templates["pepr"] = go.layout.Template(layout=dict(
    font=dict(family=FONT, size=13, color=STYLE["ink"]),
    paper_bgcolor=STYLE["surface"], plot_bgcolor=STYLE["surface"], colorway=STYLE["series"],
    title=dict(font=dict(size=16, color=STYLE["ink"]), x=0, xanchor="left", xref="paper"),
    xaxis=_axis, yaxis=_axis,
    legend=dict(font=dict(color=STYLE["ink2"]), title=dict(text="")),
    hoverlabel=dict(bgcolor=STYLE["surface"], bordercolor=STYLE["axis"], font=dict(family=FONT, color=STYLE["ink"])),
    barcornerradius=4, bargap=0.35, margin=dict(l=16, r=48, t=64, b=48),
))
pio.templates.default = "pepr"
pio.renderers.default = "plotly_mimetype+notebook_connected"


def _bars(values, unit, fmt, suffix, labels):
    values = values[::-1]
    return go.Bar(
        x=values.values, y=[str(k) for k in values.index], orientation="h", marker_color=STYLE["series"][0],
        text=[f"{v:{fmt}}{suffix}" for v in values.values] if labels else None,
        textposition="outside", textfont=dict(color=STYLE["ink2"]), cliponaxis=False, showlegend=False,
        hovertemplate=f"<b>%{{y}}</b><br>%{{x:{fmt}}}{suffix} {unit}<extra></extra>")


def hbar(values, title, unit, fmt=",d", suffix="", labels=False, x_title=None):
    fig = go.Figure(_bars(values, unit, fmt, suffix, labels))
    fig.update_layout(title=title, xaxis_title=x_title or unit, height=max(210, 28 * len(values) + 120))
    fig.update_yaxes(showgrid=False, tickfont=dict(color=STYLE["ink2"]))
    return fig


def hbar_pair(left, right, titles, units, fmt=",d"):
    fig = make_subplots(rows=1, cols=2, subplot_titles=titles, horizontal_spacing=0.3)
    for col, (values, unit) in enumerate(((left, units[0]), (right, units[1])), start=1):
        fig.add_trace(_bars(values, unit, fmt, "", False), row=1, col=col)
        fig.update_xaxes(title_text=unit, row=1, col=col)
    fig.update_yaxes(showgrid=False, tickfont=dict(color=STYLE["ink2"]))
    fig.update_annotations(font=dict(size=14, color=STYLE["ink"]))
    fig.update_layout(height=max(260, 28 * max(len(left), len(right)) + 140))
    return fig

print("Data directory:", DATA_DIR.resolve())

Data directory: /mnt/c/Users/aduvermy/Documents/hackaton-sante-numerique-2026/data/merge


## 1 — Load & flatten into the three Softcite tables

Each `*.software.json` file looks like a native Softcite output, plus two merge-specific fields:
- `merged_from` (document level): `{softcite: n, baguette: n, bio2flow: n}` — how many mentions each
  tool contributed;
- `origins` (per mention): the list of tools that produced that mention.

As in the original notebook we flatten into three tidy tables: `papers`, `mentions` and
`purpose_assessments`. We additionally keep the mention `type` (`software` or `dataset`) and the
`origins` list, and we read the document-level `merged_from` onto the `papers` table.

In [2]:
files = sorted(DATA_DIR.glob("*.software.json"))
print(f"Found {len(files)} merged files")

records = []
for f in files:
    with open(f, encoding="utf-8") as fh:
        records.append((f, json.load(fh)))


Found 265 merged files


In [3]:
def field(mention, key):
    """Return (raw, normalized) for an optional sub-object like version/publisher/url."""
    obj = mention.get(key) or {}
    return obj.get("rawForm"), obj.get("normalizedForm")


# --- papers table ---------------------------------------------------------
paper_rows = []
for f, d in records:
    paper_id = d.get("id") or f.name.replace(".software.json", "")
    biblio = d.get("biblio") or {}
    merged = d.get("merged_from") or {}
    paper_rows.append({
        "paper_id": paper_id,
        "title": biblio.get("title"),
        "authors": biblio.get("authors"),
        "doi": biblio.get("doi"),
        "n_mentions": len(d.get("mentions", [])),
        "merged_softcite": merged.get("softcite", 0),
        "merged_baguette": merged.get("baguette", 0),
        "merged_bio2flow": merged.get("bio2flow", 0),
        "file_name": f.name,
    })

papers = pd.DataFrame(paper_rows)
papers.head()

,paper_id,title,authors,doi,n_mentions,merged_softcite,merged_baguette,merged_bio2flow,file_name
0,9a6f42659f92bc42e55c72783fa433a1ce86e6b4,A Blockchain-Enhanced Reversible Watermarking ...,"Reda Bellafqira, Chloé Berton, Gouenou Coatrieux",10.1109/csp66295.2025.00016,29,11,15,3,ABlockchainEnhancedReversibleWatermarkingFrame...
1,857559d553177a7f858abc48047c6f04209429e6,AI for Computational Biology: Highlights from ...,"M Ahmad, Michał Denkiewicz, Rishika Gupta, Sac...",NaN,236,116,109,11,AIforComputationalBiologyHighlightsfromthefirs...
2,18fe244c5272cf018591691ea078a27ab937594f,A Standards-Based Knowledge Graph that Bridges...,"Marie Schmit, Ulysse Le Clanche, George Marchm...",NaN,429,376,50,3,AStandardsBasedKnowledgeGraphthatBridgesScient...
3,ffecb02ef212e3593981ae9bb28d5a16ca1c4b86,A Survey on Multimodal Federated Learning,"Silvia Grosso, Nawel Benarba, Sara Bouchenak, ...",NaN,96,49,9,38,ASurveyonMultimodalFederatedLearning.software....
4,72bd546fbc5c44c0be94925d3ac67eff54c9171c,NaN,NaN,NaN,23,2,5,16,AWhiteBoxWatermarkingModulationforEncryptedDNN...


In [4]:
SCOPES = {"local": "mentionContextAttributes", "document": "documentContextAttributes"}
PURPOSES = ("used", "created", "shared")


def melt_purposes(mention, paper_id, mention_index):
    """Unpivot a mention's two attribute blocks into 6 tidy purpose-assessment rows."""
    sid = f"{paper_id}:tei:{mention_index}"
    rows = []
    for scope, block_key in SCOPES.items():
        attrs = mention.get(block_key) or {}
        for purpose in PURPOSES:
            a = attrs.get(purpose) or {}
            rows.append({
                "software_mention_id": sid,
                "paper_id": paper_id,
                "mention_index": mention_index,
                "scope": scope,
                "purpose": purpose,
                "value": bool(a.get("value", False)),
                "certainty_score": float(a["score"]) if "score" in a else float("nan"),
            })
    return rows


first_mention = next(m for _f, _d in records for m in _d.get("mentions", []))
pd.DataFrame(melt_purposes(first_mention, "DEMO", 0))

,software_mention_id,paper_id,mention_index,scope,purpose,value,certainty_score
0,DEMO:tei:0,DEMO,0,local,used,False,9.250641e-05
1,DEMO:tei:0,DEMO,0,local,created,False,5.209446e-05
2,DEMO:tei:0,DEMO,0,local,shared,False,2.384186e-07
3,DEMO:tei:0,DEMO,0,document,used,True,9.998484e-01
4,DEMO:tei:0,DEMO,0,document,created,False,1.341581e-03
5,DEMO:tei:0,DEMO,0,document,shared,False,3.576279e-07


In [5]:
# --- mentions & purpose_assessments tables --------------------------------
mention_rows = []
purpose_rows = []
for f, d in records:
    paper_id = d.get("id") or f.name.replace(".software.json", "")
    for i, m in enumerate(d.get("mentions", [])):
        name = m.get("software-name") or {}
        lang = m.get("language") or {}
        v_raw, v_norm = field(m, "version")
        p_raw, p_norm = field(m, "publisher")
        u_raw, u_norm = field(m, "url")
        mention_rows.append({
            "software_mention_id": f"{paper_id}:tei:{i}",
            "paper_id": paper_id,
            "mention_index": i,
            "type": m.get("type", "software"),
            "software_raw": name.get("rawForm"),
            "software_normalized": name.get("normalizedForm"),
            "software_type": m.get("software-type"),
            "origins": m.get("origins", []),
            "version_raw": v_raw,
            "version_normalized": v_norm,
            "publisher_raw": p_raw,
            "publisher_normalized": p_norm,
            "language_raw": lang.get("rawForm"),
            "language_normalized": lang.get("normalizedForm"),
            "language_wikidata": lang.get("wikidataId"),
            "url_raw": u_raw,
            "url_normalized": u_norm,
            "context_full_text": m.get("context"),
            "n_references": len(m.get("references") or []),
        })
        purpose_rows.extend(melt_purposes(m, paper_id, i))

mentions = pd.DataFrame(mention_rows)
purpose_assessments = pd.DataFrame(purpose_rows)
mentions.head()

,software_mention_id,paper_id,mention_index,type,software_raw,software_normalized,software_type,origins,version_raw,version_normalized,publisher_raw,publisher_normalized,language_raw,language_normalized,language_wikidata,url_raw,url_normalized,context_full_text,n_references
0,9a6f42659f92bc42e55c72783fa433a1ce86e6b4:tei:0,9a6f42659f92bc42e55c72783fa433a1ce86e6b4,0,software,Alg,Alg,software,[softcite],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Alg. 2 ensures perfect reconstruction of the o...,0
1,9a6f42659f92bc42e55c72783fa433a1ce86e6b4:tei:1,9a6f42659f92bc42e55c72783fa433a1ce86e6b4,1,software,VGG16 2,VGG16 2,software,[softcite],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,We utilized the VGG16 2 architecture as our ba...,0
2,9a6f42659f92bc42e55c72783fa433a1ce86e6b4:tei:2,9a6f42659f92bc42e55c72783fa433a1ce86e6b4,2,software,VGG16,VGG16,software,[softcite],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,We utilized a pre-trained VGG16 network fine-t...,0
3,9a6f42659f92bc42e55c72783fa433a1ce86e6b4:tei:3,9a6f42659f92bc42e55c72783fa433a1ce86e6b4,3,software,VGG16,VGG16,software,[softcite],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,We trained two versions of the VGG16 model: i)...,0
4,9a6f42659f92bc42e55c72783fa433a1ce86e6b4:tei:4,9a6f42659f92bc42e55c72783fa433a1ce86e6b4,4,software,VGG16,VGG16,software,[softcite],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Testing on a medical dataset using a VGG16 mod...,0


In [6]:
# --- sanity checks (the invariants the Softcite schema guarantees) ----------
print(f"papers:              {papers.shape}")
print(f"mentions:            {mentions.shape}")
print(f"purpose_assessments: {purpose_assessments.shape}")

assert len(purpose_assessments) == 6 * len(mentions), "expected exactly 6 purpose rows per mention"
assert mentions["software_mention_id"].is_unique, "software_mention_id must be unique"
assert (purpose_assessments.groupby("software_mention_id").size() == 6).all()
print("\nAll invariants hold ✅")

print("\nMention types:")
print(mentions["type"].value_counts().to_string())
print("\nOrigins of the mentions:")
print(mentions["origins"].map(lambda o: ", ".join(o)).value_counts().to_string())

papers:              (265, 9)
mentions:            (18287, 19)
purpose_assessments: (109722, 7)

All invariants hold ✅

Mention types:
type
software    14357
dataset      3930

Origins of the mentions:
origins
softcite    12115
baguette     5137
bio2flow     1035


### Explicit and implicit mentions, software vs dataset

The merged data mixes the outputs of three tools, so a few conventions are needed:

- **type** distinguishes named software from `dataset` mentions (Baguette labels many references to
  data collections as `dataset`). Sections 2–6 focus on `type == "software"`; datasets are analysed
  separately in section 7.
- **software-type** has a heterogeneous meaning across sources (Softcite/Bio2Flow call named tools
  `software`/`component`/`environment` and generic references `implicit`; Baguette used
  `software`/`implicit` to mean explicit/implicit). We keep the field as-is and only use it for the
  rough explicit/implicit split below.

From here on, `mentions`, `purpose_assessments` and `software` hold the **explicit** `software`
mentions only. The implicit ones are kept aside for section 8; the datasets are kept in
`dataset_mentions` for section 7.

In [7]:
all_software_mentions = mentions
is_implicit = all_software_mentions["software_type"].eq("implicit")

implicit_mentions = all_software_mentions[is_implicit].copy()
implicit_purposes = purpose_assessments[
    purpose_assessments["software_mention_id"].isin(implicit_mentions["software_mention_id"])]

# Analyses of sections 2–6 use the explicit software mentions only.
mentions = all_software_mentions[~is_implicit].copy()
purpose_assessments = purpose_assessments[
    purpose_assessments["software_mention_id"].isin(mentions["software_mention_id"])]

# Datasets are handled separately in section 7.
dataset_mentions = all_software_mentions[all_software_mentions["type"] == "dataset"].copy()
dataset_purposes = purpose_assessments[
    purpose_assessments["software_mention_id"].isin(dataset_mentions["software_mention_id"])]

# Recompute counts for the software-only tables.
papers["n_mentions"] = papers["paper_id"].map(mentions.groupby("paper_id").size()).fillna(0).astype(int)
papers["n_datasets"] = papers["paper_id"].map(dataset_mentions.groupby("paper_id").size()).fillna(0).astype(int)

print("explicit software mentions:", len(mentions), "in", int((papers["n_mentions"] > 0).sum()), "papers")
print("implicit software mentions:", len(implicit_mentions))
print("dataset mentions:          ", len(dataset_mentions))
print()
print(all_software_mentions["software_type"].value_counts().to_string())

explicit software mentions: 15676 in 252 papers
implicit software mentions: 2611
dataset mentions:           3930

software_type
software       14297
implicit        2611
component        205
environment      139


### One row per software and paper

Sections 2–6 work on the **software of each paper**, not on the mentions: a software named several
times in a paper counts once. The `software` table has one row per paper and normalized name, with
`used`/`created`/`shared` taken from the **document-level** characterization
(`documentContextAttributes`). We also record which tool first produced each
(paper, software) pair via `origin` (the first element of `origins`).

In [8]:
doc_scope = purpose_assessments[purpose_assessments["scope"] == "document"]
doc_values = doc_scope.pivot(index="software_mention_id", columns="purpose", values="value")
doc_scores = doc_scope.pivot(index="software_mention_id", columns="purpose", values="certainty_score").add_suffix("_score")
with_doc = mentions.join(doc_values, on="software_mention_id").join(doc_scores, on="software_mention_id")

keys = ["paper_id", "software_normalized"]
assert (with_doc.groupby(keys)[list(PURPOSES)].nunique() <= 1).all().all(), "document values differ within a paper"

any_ = lambda s: s.notna().any()
software = (with_doc.groupby(keys, as_index=False)
            .agg(n_mentions=("software_mention_id", "size"),
                 origin=("origins", lambda s: s.iloc[0][0] if len(s.iloc[0]) else None),
                 **{p: (p, "first") for p in PURPOSES},
                 **{f"{p}_score": (f"{p}_score", "first") for p in PURPOSES},
                 has_version=("version_normalized", any_),
                 has_publisher=("publisher_normalized", any_),
                 has_language=("language_normalized", any_),
                 has_url=("url_normalized", any_),
                 has_citation=("n_references", lambda s: (s > 0).any()),
                 language=("language_normalized", "first"),
                 publisher=("publisher_normalized", "first")))

papers["n_software"] = papers["paper_id"].map(software.groupby("paper_id").size()).fillna(0).astype(int)
print(f"software: {len(software)} (paper, software) rows, {software['software_normalized'].nunique()} distinct normalized names, "
      f"from {len(mentions)} explicit mentions")
software.head()

software: 6099 (paper, software) rows, 4892 distinct normalized names, from 15676 explicit mentions


,paper_id,software_normalized,n_mentions,origin,used,created,shared,used_score,created_score,shared_score,has_version,has_publisher,has_language,has_url,has_citation,language,publisher
0,023222d011f63def7a90b3e720e9b036d0591c5c,BRAT,1,baguette,True,False,False,1.000000,0.000000,0.000000,False,False,False,False,False,NaN,NaN
1,023222d011f63def7a90b3e720e9b036d0591c5c,BRAT-Eval,1,softcite,False,False,False,0.177453,0.000005,0.000003,False,False,False,False,False,NaN,NaN
2,023222d011f63def7a90b3e720e9b036d0591c5c,BioToFlow,1,baguette,True,False,False,1.000000,0.000000,0.000000,False,False,False,False,False,NaN,NaN
3,023222d011f63def7a90b3e720e9b036d0591c5c,Bioinformatics,1,baguette,False,False,False,0.000000,0.000000,0.000000,False,False,False,False,False,NaN,NaN
4,023222d011f63def7a90b3e720e9b036d0591c5c,F1000Research,1,baguette,False,False,False,0.000000,0.000000,0.000000,False,False,False,False,False,NaN,NaN


## 2 — Dataset overview

In [9]:
overview = pd.Series({
    "papers": len(papers),
    "papers with >=1 explicit software": int((papers["n_software"] > 0).sum()),
    "papers with >=1 dataset": int((papers["n_datasets"] > 0).sum()),
    "papers with biblio (title)": int(papers["title"].notna().sum()),
    "papers with DOI": int(papers["doi"].notna().sum()),
    "distinct software (normalized names)": software["software_normalized"].nunique(),
    "(paper, software) pairs": len(software),
    "explicit software mentions": len(mentions),
    "dataset mentions": len(dataset_mentions),
    "implicit software mentions (section 8)": len(implicit_mentions),
})
rate = (papers["n_software"] > 0).mean()
print(overview.to_string())
print(f"\nShare of papers naming at least one software: {rate:.1%}")
print(f"Software per paper, among those papers: median {papers.loc[papers['n_software'] > 0, 'n_software'].median():.0f}, "
      f"mean {papers.loc[papers['n_software'] > 0, 'n_software'].mean():.1f}")

papers                                      265
papers with >=1 explicit software           252
papers with >=1 dataset                     239
papers with biblio (title)                  236
papers with DOI                             114
distinct software (normalized names)       4892
(paper, software) pairs                    6099
explicit software mentions                15676
dataset mentions                           3930
implicit software mentions (section 8)     2611

Share of papers naming at least one software: 95.1%
Software per paper, among those papers: median 18, mean 24.2


In [10]:
# Distribution of distinct software per paper
per_paper = papers["n_software"].value_counts().reindex(range(papers["n_software"].max() + 1), fill_value=0)
fig = go.Figure(go.Bar(
    x=per_paper.index, y=per_paper.values, marker=dict(color=STYLE["series"][0], line=dict(color=STYLE["surface"], width=1)),
    hovertemplate="%{y} papers name %{x} distinct software<extra></extra>"))
fig.update_layout(title="Explicit software per paper (distinct normalized names)", bargap=0, height=380,
                  xaxis_title="distinct software named in a paper", yaxis_title="papers")
fig.update_xaxes(showgrid=False)
fig.show()

print(papers["n_software"].describe().to_string())

count    265.000000
mean      23.015094
std       24.310798
min        0.000000
25%        6.000000
50%       17.000000
75%       29.000000
max      177.000000


## 3 — Per-source and complementarity analysis

The merge keeps the native Softcite mentions as the reference and **adds** the mentions that
Baguette and Bio2Flow detected but Softcite missed. This section shows the value of merging:

- how many mentions and (paper, software) pairs each tool contributed;
- how many software Baguette/Bio2Flow add on top of Softcite alone;
- how much each tool alone would have found.

`merged_from` (document level) counts mentions per source; `origin` (first tool that produced a
software in a paper) lets us count software per source.

In [11]:
# How many (paper, software) pairs each tool is the *first* (and only, here) origin of
origin_counts = software["origin"].value_counts()
origin_counts = origin_counts.reindex(["softcite", "baguette", "bio2flow"]).fillna(0).astype(int)
print("(paper, software) pairs by origin tool:")
print(origin_counts.to_string())
print(f"\nShare contributed by Softcite alone: {origin_counts['softcite'] / len(software):.1%}")
print(f"Added by Baguette + Bio2Flow over Softcite: {origin_counts['baguette'] + origin_counts['bio2flow']} "
      f"({(origin_counts['baguette'] + origin_counts['bio2flow']) / len(software):.1%})")

(paper, software) pairs by origin tool:
origin
softcite    2242
baguette    2822
bio2flow    1035

Share contributed by Softcite alone: 36.8%
Added by Baguette + Bio2Flow over Softcite: 3857 (63.2%)


In [12]:
# Mentions contributed per source (document-level merged_from)
by_source = papers[["merged_softcite", "merged_baguette", "merged_bio2flow"]].sum()
hbar(by_source, "Mentions contributed per tool (sum of merged_from)", "mentions").show()
by_source

merged_softcite    12115
merged_baguette     5137
merged_bio2flow     1035
dtype: int64

In [13]:
# Software added by each other tool over Softcite alone, by papers
added = software[software["origin"] != "softcite"]
by_papers = added["software_normalized"].value_counts().head(20)
if len(by_papers):
    hbar(by_papers, "Top software detected ONLY by Baguette/Bio2Flow (not Softcite), by papers", "papers").show()
else:
    print("No software detected only by Baguette/Bio2Flow.")
by_papers

software_normalized
R               37
GitHub          11
CIFAR-10        10
Zenodo           9
CIFAR10          8
θ                7
MNIST            7
ML               6
SHAP             6
KEGG             6
DP               5
BERT             5
GPT              5
ImageNet         5
SAEM             5
BIC              5
MAP-BE           5
GAN              4
UK Biobank       4
Hugging Face     4
Name: count, dtype: int64

In [14]:
# How many distinct software each tool ALONE would have found across the corpus
per_tool = software.groupby("origin")["software_normalized"].nunique()
print("Distinct software detected per tool (as first origin):")
print(per_tool.to_string())
print(f"\nTotal distinct software in the merged data: {software['software_normalized'].nunique()}")

Distinct software detected per tool (as first origin):
origin
baguette    2522
bio2flow     937
softcite    1620

Total distinct software in the merged data: 4892


## 4 — Top software

In [15]:
TOP_N = 20

by_papers = software["software_normalized"].value_counts().head(TOP_N)
hbar(by_papers, f"Top {TOP_N} explicit software by papers", "papers").show()
by_papers

software_normalized
Python          43
R               37
Nextflow        18
Adam            18
Snakemake       16
GitHub          14
PyTorch         14
Docker          13
code            12
python          11
XGBoost         11
FedAvg          11
Galaxy          11
scikit-learn    11
DESeq2          10
CIFAR-10        10
Zenodo           9
Seurat           9
ChatGPT          9
Monolix          9
Name: count, dtype: int64

## 5 — Purpose: the document characterization (the core analysis)

How many (paper, software) pairs are characterized as **used**, **created** or **shared** at the
document level.

In [16]:
# How many (paper, software) pairs are characterized as used / created / shared.
assert all(((software[f"{p}_score"] > 0.5) == software[p]).all() for p in PURPOSES)
characterized = pd.Series({p: int(software[p].sum()) for p in PURPOSES}, name="(paper, software) pairs")
share = characterized / len(software) * 100

fig = go.Figure(go.Bar(
    x=share.values[::-1], y=list(share.index[::-1]), orientation="h", marker_color=STYLE["series"][0],
    customdata=characterized.values[::-1], text=[f"{v:.1f}%" for v in share.values[::-1]],
    textposition="outside", textfont=dict(color=STYLE["ink2"]), cliponaxis=False,
    hovertemplate="<b>%{y}</b><br>%{customdata:,d} (paper, software) pairs, %{x:.1f}%<extra></extra>"))
fig.update_layout(title=f"Document characterization of the {len(software):,} (paper, software) pairs",
                  xaxis_title="% of (paper, software) pairs", height=210)
fig.update_xaxes(range=[0, 100])
fig.update_yaxes(showgrid=False, tickfont=dict(color=STYLE["ink2"]))
fig.show()
characterized

used       3685
created     589
shared      143
Name: (paper, software) pairs, dtype: int64

In [17]:
def top_software_for(purpose, min_score=0.5, top_n=15):
    """Papers per software where the document characterization says `purpose` with score > min_score."""
    hits = software[software[purpose] & (software[f"{purpose}_score"] > min_score)]
    return hits["software_normalized"].value_counts().head(top_n)


popular_used = top_software_for("used")
hbar(popular_used, "Most-used explicit software (document characterization, score > 0.5)", "papers").show()
popular_used

software_normalized
R               37
Python          31
Adam            16
Nextflow        13
PyTorch         11
DESeq2           9
XGBoost          9
GitHub           9
python           9
Snakemake        9
Monolix          9
code             9
FedAvg           8
ImageNet         8
scikit-learn     8
Name: count, dtype: int64

In [18]:
print("Top software CREATED (document characterization, score > 0.5), by papers:")
print(top_software_for("created").to_string())
print("\nTop software SHARED (document characterization, score > 0.5), by papers:")
print(top_software_for("shared").to_string() or "(none above threshold)")

Top software CREATED (document characterization, score > 0.5), by papers:
software_normalized
Nextflow       12
Python         10
FedAvg          6
Snakemake       6
code            6
XGBoost         4
Galaxy          4
scripts         4
BoNesis         3
EDAM            3
PyTorch         3
Hyperledger     3
Docker          3
Mapper          3
PROV graph      2

Top software SHARED (document characterization, score > 0.5), by papers:
software_normalized
code            5
Python          4
NeurIPS         4
NLStruct        2
gitOmmix        2
MultiQC         2
Software        2
Nextflow        2
NeuroConText    2
ktest           2
medkit          2
BoNesis         2
AstroLogics     1
function        1
scripts         1


## 6 — Metadata richness

In [19]:
richness = pd.Series({
    "has version": software["has_version"].mean(),
    "has publisher": software["has_publisher"].mean(),
    "has language": software["has_language"].mean(),
    "has url": software["has_url"].mean(),
    "has >=1 citation": software["has_citation"].mean(),
}) * 100
hbar(richness.sort_values(ascending=False), f"Share of the {len(software):,} (paper, software) pairs carrying each attribute",
     "of the (paper, software) pairs", fmt=".1f", suffix="%", labels=True,
     x_title="% of (paper, software) pairs").show()
richness.round(1)

has version         4.5
has publisher       0.7
has language        1.2
has url             3.2
has >=1 citation    5.1
dtype: float64

## 7 — Dataset analysis (a separate look at the `dataset` mentions)

Baguette labels references to data collections as `dataset` (e.g. `UK Biobank`, `MIMIC-III`).
They are not software, but they matter for research-practice analysis, so we keep them apart. This
section gives a simple, pragmatic overview.

In [20]:
print(f"dataset mentions: {len(dataset_mentions)} across {dataset_mentions['paper_id'].nunique()} papers "
      f"({dataset_mentions['paper_id'].nunique() / len(papers):.1%} of the papers)")
print("\nTop dataset mentions by papers:")
top_datasets = (dataset_mentions.assign(form=dataset_mentions["software_normalized"].str.lower().str.strip())
                .groupby("form")["paper_id"].nunique().sort_values(ascending=False).head(15))
print(top_datasets.to_string())
hbar(top_datasets, "Top datasets mentioned, by papers", "papers").show()

dataset mentions: 3930 across 239 papers (90.2% of the papers)

Top dataset mentions by papers:
form
dataset             50
data                25
corpus              15
synthetic data      14
simulated data      14
test set            13
training set        11
cifar-10            10
training dataset    10
test dataset         9
github               9
rna-seq data         8
cifar10              8
original dataset     8
datasets             8


In [21]:
# How many dataset mentions each tool contributed (their origin)
origin = dataset_mentions["origins"].map(lambda o: ", ".join(o)).value_counts()
print("dataset mentions by origin:")
print(origin.to_string())
hbar(origin, "Dataset mentions by origin tool", "mentions").show()

dataset mentions by origin:
origins
baguette    3930


In [22]:
# Datasets per paper
print("Datasets per paper (distribution):")
print(papers.loc[papers["n_datasets"] > 0, "n_datasets"].describe().round(1).to_string())

# How often datasets are used / created / shared at document level (if any assessment fired)
if len(dataset_mentions):
    ds_doc = purpose_assessments[purpose_assessments["scope"] == "document"]
    ds_doc = ds_doc[ds_doc["software_mention_id"].isin(dataset_mentions["software_mention_id"])]
    fired = (ds_doc[ds_doc["value"]].groupby("purpose")["software_mention_id"].nunique())
    print("\nDataset mentions flagged used/created/shared at document level:")
    print(fired.reindex(list(PURPOSES)).fillna(0).astype(int).to_string())

Datasets per paper (distribution):
count    239.0
mean      16.4
std       17.1
min        1.0
25%        5.0
50%       13.0
75%       21.0
max       99.0

Dataset mentions flagged used/created/shared at document level:
purpose
used       1515
created     142
shared        0


## 8 — Single-software deep dive

In [23]:
# Default to the software named in the most papers, so the cell always has something to show.
SOFTWARE = software["software_normalized"].value_counts().index[0]

rows = software[software["software_normalized"] == SOFTWARE]
print(f"Software: {SOFTWARE!r}")
print(f"  papers:   {len(rows)}")
print(f"  mentions: {int(rows['n_mentions'].sum())}")
print("  document characterization (papers): " + ", ".join(f"{p} {int(rows[p].sum())}" for p in PURPOSES))
print("\nSample contexts:")
for ctx in mentions.loc[mentions["software_normalized"] == SOFTWARE, "context_full_text"].dropna().head(3):
    print("  •", (ctx[:240] + "…") if len(ctx) > 240 else ctx)

Software: 'Python'
  papers:   43
  mentions: 127
  document characterization (papers): used 31, created 10, shared 4

Sample contexts:
  • Finally, the team plans to containerize the pipeline as a Python package, which mirrors Team 1's and Team 4's emphasis on reproducibility and usability, pointing toward a future in which multi-omic data analysis can be deployed modularly ac…
  • Team 7 proposed packaging their work into a standardized and reusable Python module, enabling wider dissemination and reproducibility.
  • Expression patterns of phase-determining genes were visualized through ridge plots using Seaborn 0.13.2 visualization libraries in Python 3.11.


## 9 — Implicit mentions

Implicit mentions refer to software without naming it ("our code", "the scripts", "a tool"). They
say little about *which* software, so we only look at their generic forms and their purposes.

In [24]:
implicit_forms = implicit_mentions.assign(form=implicit_mentions["software_normalized"].str.lower().str.strip())

print(f"{len(implicit_mentions)} implicit mentions (not matched to a software) in {implicit_mentions['paper_id'].nunique()} papers "
      f"({implicit_mentions['paper_id'].nunique() / len(papers):.1%} of the papers), "
      f"{implicit_forms['form'].nunique()} distinct forms")

forms_by_mentions = implicit_forms["form"].value_counts().head(TOP_N)
forms_by_papers = (implicit_forms.groupby("form")["paper_id"].nunique()
                   .sort_values(ascending=False).head(TOP_N))

hbar_pair(forms_by_mentions, forms_by_papers,
          (f"Top {TOP_N} implicit forms by mentions", f"Top {TOP_N} implicit forms by papers"),
          ("mentions", "papers")).show()

2611 implicit mentions (not matched to a software) in 234 papers (88.3% of the papers), 1893 distinct forms


In [25]:
# Purposes flagged True for the implicit mentions, at mention (local) and document scope
implicit_fired = (implicit_purposes[implicit_purposes["value"]]
                  .groupby(["scope", "purpose"]).size().unstack("purpose")
                  .reindex(index=["local", "document"], columns=list(PURPOSES)).fillna(0).astype(int))


def share_fired(assessments, n):
    return (assessments[assessments["value"]].groupby(["scope", "purpose"]).size() / n * 100).unstack("purpose") \
        .reindex(index=["local", "document"], columns=list(PURPOSES)).fillna(0)


SCOPE_LABELS = {"local": "mention (local)", "document": "document"}
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.06,
                    subplot_titles=(f"Implicit mentions (n={len(implicit_mentions):,})",
                                    f"Explicit mentions (n={len(mentions):,})"))
for col, (assessments, n) in enumerate(((implicit_purposes, len(implicit_mentions)),
                                        (purpose_assessments, len(mentions))), start=1):
    share = share_fired(assessments, n)
    for k, purpose in enumerate(PURPOSES):
        fig.add_trace(go.Bar(
            x=share[purpose].values, y=[SCOPE_LABELS[s] for s in share.index], orientation="h",
            name=purpose, legendgroup=purpose, showlegend=col == 1, marker_color=STYLE["series"][k],
            hovertemplate=f"<b>{purpose}</b> · %{{y}}<br>%{{x:.1f}}% of the mentions<extra></extra>"), row=1, col=col)
fig.update_layout(title="Purposes flagged for each kind of mention, by scope", barmode="group", bargroupgap=0.08,
                  height=360, margin=dict(t=96), legend=dict(traceorder="normal"))
fig.update_xaxes(title_text="% of mentions flagged True", range=[0, 100])
fig.update_yaxes(showgrid=False, tickfont=dict(color=STYLE["ink2"]), autorange="reversed")
fig.update_annotations(font=dict(size=14, color=STYLE["ink"]))
fig.show()
implicit_fired

purpose,used,created,shared
scope,,,
local,1390,933,86
document,1452,971,112


## 10 — Limitations

- **No publication year / genre / license.** The merged files carry only `biblio` (title, authors,
  doi), so a per-year or per-genre breakdown is not possible from this data alone.
- **Probabilistic extractions.** The mentions come from ML models (Softcite f-scores around 0.8;
  Baguette and Bio2Flow are predictions too), so expect both false positives and false negatives.
  The `certainty_score` thresholds (e.g. `> 0.5`) trade recall for precision.
- **Heterogeneous sources.** The merged data mixes three tools with different conventions:
  - `software-type` means different things: Softcite/Bio2Flow tag named tools as
    `software`/`component`/`environment` and generic references as `implicit`, while Baguette used
    `software`/`implicit` to mean explicit/implicit. Some Bio2Flow mentions have `software-type == null`.
  - Baguette has no offsets (`offsetStart`/`offsetEnd` are null).
  - Baguette also labels many references to data collections as `dataset`, which is why datasets are
    analysed separately (section 7).
- **Normalized names.** Software is identified by the `normalizedForm`, which is not case-folded, so
  e.g. `TensorFlow`/`Tensorflow` may be counted twice in a paper. Some entries are extraction noise
  rather than named software (algorithms such as `Adam`, file names such as `graph.py`).
- **`origin` is the first tool, not a vote.** In this merge (union without dedup), a software appears
  in a paper under one origin — the first tool that produced a mention with that name. The same
  software is often detected by several tools, so `origin` understates multi-tool agreement.
- **Document characterization.** Sections 5–6 use the document-level `used`/`created`/`shared`,
  which is the same for every mention of a software in a paper. The mention-level (`local`)
  assessments are only used in section 9.